# 03 — scikit-learn Baseline

This is the core methodology module: everything from here on assumes we
have ground-truth ratings to learn from and evaluate against. Nothing in
Notebooks 01-02 involved a human judgment call about *quality* — this is
where that happens.

**Part A (this session): hand-labeling.** You rate each candidate 1-5 on
"would I actually spot this from the air" using the FAA VFR sectional chart -- what a pilot actually flies with -- not
a guess from the tabular features. This has to be done interactively in a
live Jupyter kernel — run the labeling cell below yourself; it isn't
something that can be scripted or executed headlessly, since the whole
point is your visual judgment.

(2026-08-30: relabeled against the VFR sectional chart, replacing an earlier satellite-imagery pass -- see data/labels/spottability_ratings_satellite_v1.csv.bak for why; labeling now happens on the chart at /app/label.)

**Part B (once labels exist):** dummy baseline, k-fold cross-validation,
model comparison (Ridge/RandomForest/GradientBoosting), hyperparameter
tuning, bias/variance diagnostics (learning curves, regularization sweep),
final held-out evaluation, and permutation importance. That part isn't
built yet — it depends on seeing the real label distribution first (e.g.
whether ratings skew toward 3-4, how many 1s/2s show up, class balance),
so it'll be added after you've labeled a batch.


In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT / "src"))

import pandas as pd

pd.set_option("display.max_columns", None)


## Step 1 — Load the feature table

This is Notebook 02's output — 230 candidates with route geometry, size,
elevation prominence, name uniqueness, and clutter distance already
computed. Labeling doesn't use those numbers directly (that would be
circular — the whole point of Module 03 is learning a mapping from these
features to your rating), just `name`/`category`/`lat`/`lon` to show you
each candidate.


In [ ]:
FEATURES_PATH = PROJECT_ROOT / "data" / "processed" / "features_c81_kdlh.parquet"
LABELS_PATH = PROJECT_ROOT / "data" / "labels" / "spottability_ratings.csv"

candidates_df = pd.read_parquet(FEATURES_PATH)
print(f"{len(candidates_df)} candidates available to label")
candidates_df[["name", "category", "along_track_nm"]].head()


## Step 2 — Hand-label

Run the cell below. For each candidate it shows the FAA VFR sectional chart centered
on its coordinates and asks for a rating:

- **1-5**: your "would I actually spot this from the air" call (1 = no
  chance, 5 = unmissable)
- **blank + Enter**: skip for now, it'll come back on a future run
- **q**: stop the session — whatever you've rated so far is already saved

Candidates are shown in a shuffled (but reproducible) order rather than
along-route, so if you stop partway through you won't have systematically
skipped an entire region of the route. Target range from the plan is
**50-150 labels** — you don't need to do all 230, and you can run this
cell across multiple sessions (it skips whatever's already in
`data/labels/spottability_ratings.csv`).


In [ ]:
# Labeling moved out of the notebook. It now happens on the chart
# itself at /app/label (webapp, port 8080), which reads the FAA
# sectional directly rather than rating candidates OpenStreetMap
# supplied -- see data/labels/chart_picks.csv.


## Step 3 — Check what you've got so far

Once you've labeled a batch, run this to see the count and rating
distribution. Come back to Notebook 03 (Part B gets added once there's a
real distribution to build the CV/model-comparison steps against) after
you've hit at least the low end of the 50-150 target.


In [ ]:
if LABELS_PATH.exists():
    labels_df = pd.read_csv(LABELS_PATH)
    print(f"{len(labels_df)} candidates labeled so far")
    print(labels_df["rating"].value_counts().sort_index())
else:
    print("No labels yet -- run the Step 2 cell first.")


## Part B — learning from your labels

228 candidates got labeled (target was 50-150), so there's plenty of
signal. The distribution is skewed toward the easy end -- **139 fives**,
with the remaining candidates spread fairly evenly across 1-4 (30/22/13/24)
-- so a plain 80/20 random split risks putting almost all the 1s and 2s in
one side. Everything below stratifies on rating to avoid that.

Rating is treated as a continuous 1-5 **regression** target (not a 5-class
classifier) -- "spottability" is inherently ordinal (a 4 is closer to a 5
than to a 1), and regression is what makes the regularization-sweep /
bias-variance framing from the plan (Ridge alpha) meaningful.

In [ ]:
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.inspection import permutation_importance
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import (
    GridSearchCV,
    KFold,
    cross_val_score,
    learning_curve,
    train_test_split,
    validation_curve,
)

import matplotlib.pyplot as plt
import numpy as np

RANDOM_STATE = 42


## Step 4 — Build the modeling table

Join Notebook 02's feature table to your labels on `(osm_id, osm_type)`.
Feature columns are everything numeric/engineered -- geometry
(`cross_track_nm`, `along_track_nm`, `within_preferred_corridor`), size
(`log_size`), elevation (`elevation_prominence_m`), name distinctiveness
(`name_uniqueness`), clutter (`nn_dist_nm`), and the one-hot `category_*`
columns. `name`/`lat`/`lon`/`osm_id`/`osm_type`/`category` are identifiers
shown during labeling, not modeling inputs. `name_uniqueness` is NaN for unnamed candidates (143 of 230) by design -- filled to 0.0 below, below the minimum a named-but-shared name can score.

In [ ]:
labels_df = pd.read_csv(LABELS_PATH)
# osm_id round-trips as str through the features parquet but as int64
# through a freshly-read labels CSV -- normalize both before merging
# (same fix already applied in src/vfr/pipeline.py's _load_labeled()).
candidates_df["osm_id"] = candidates_df["osm_id"].astype(str)
labels_df["osm_id"] = labels_df["osm_id"].astype(str)
labeled_df = candidates_df.merge(labels_df[["osm_id", "osm_type", "rating"]], on=["osm_id", "osm_type"])

# cross_track_nm/along_track_nm/within_preferred_corridor dropped
# 2026-09-10 as route-position leakage -- see src/vfr/pipeline.py's
# own FEATURE_COLS_BASE comment for the measured MAE numbers.
FEATURE_COLS_BASE = [
    "log_size", "elevation_prominence_m", "name_uniqueness", "nn_dist_nm",
]
# Category one-hots are derived from the parquet rather than listed by
# hand: the candidate categories change as the data-quality work
# continues (towers, water towers and quarries have all been dropped),
# and a hardcoded list silently goes stale and then KeyErrors.
FEATURE_COLS = FEATURE_COLS_BASE + [
    c for c in candidates_df.columns if c.startswith("category_")
]

X = labeled_df[FEATURE_COLS].fillna({"name_uniqueness": 0.0})
y = labeled_df["rating"].astype(float)

print(f"{len(labeled_df)} labeled candidates, {len(FEATURE_COLS)} features")
X.head()


## Step 5 — Held-out split

20% held out for the final evaluation in Step 9, stratified on rating so
the rare 1s/2s/3s aren't all left in (or out of) the training set. This
split is untouched by everything else -- CV, tuning, and diagnostics all
happen on `X_train`/`y_train` only.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

cv = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

print(f"train: {len(X_train)}, test: {len(X_test)}")
print("train rating distribution:")
print(y_train.value_counts().sort_index())
print("test rating distribution:")
print(y_test.value_counts().sort_index())


## Step 6 — Dummy baseline

Any real model needs to beat "always predict the mean rating." With
ratings this skewed toward 5, the mean is a deceptively strong baseline --
worth seeing the number before getting excited about a model's score.

In [ ]:
dummy = DummyRegressor(strategy="mean")
dummy_mae = -cross_val_score(dummy, X_train, y_train, cv=cv, scoring="neg_mean_absolute_error")
dummy_rmse = -cross_val_score(dummy, X_train, y_train, cv=cv, scoring="neg_root_mean_squared_error")

print(f"Dummy (mean) baseline -- MAE: {dummy_mae.mean():.3f} +/- {dummy_mae.std():.3f}")
print(f"Dummy (mean) baseline -- RMSE: {dummy_rmse.mean():.3f} +/- {dummy_rmse.std():.3f}")


## Step 7 — Model comparison

Three model families with different bias/variance profiles: Ridge (linear,
high bias/low variance), RandomForest (low bias, controlled variance via
averaging), GradientBoosting (low bias, prone to overfitting without
tuning). 5-fold CV on the training set only, scored on MAE (directly
interpretable: "average rating points off").

In [ ]:
models = {
    "Ridge": Ridge(random_state=RANDOM_STATE),
    "RandomForest": RandomForestRegressor(random_state=RANDOM_STATE),
    "GradientBoosting": GradientBoostingRegressor(random_state=RANDOM_STATE),
}

results = {}
for name, model in models.items():
    mae = -cross_val_score(model, X_train, y_train, cv=cv, scoring="neg_mean_absolute_error")
    results[name] = mae
    print(f"{name:>17}: MAE {mae.mean():.3f} +/- {mae.std():.3f}")

print(f"{'Dummy (mean)':>17}: MAE {dummy_mae.mean():.3f} +/- {dummy_mae.std():.3f}")


## Step 8 — Hyperparameter tuning

Grid search each model family's key knobs against the same 5-fold CV
split, scored on MAE. `Ridge`'s `alpha` controls L2 regularization
strength; the RandomForest/GradientBoosting grids trade off depth/number of
estimators against overfitting.

In [ ]:
param_grids = {
    "Ridge": (Ridge(random_state=RANDOM_STATE), {"alpha": [0.01, 0.1, 1.0, 10.0, 100.0]}),
    "RandomForest": (
        RandomForestRegressor(random_state=RANDOM_STATE),
        {"n_estimators": [100, 300], "max_depth": [3, 5, 10, None], "min_samples_leaf": [1, 3, 5]},
    ),
    "GradientBoosting": (
        GradientBoostingRegressor(random_state=RANDOM_STATE),
        {"n_estimators": [100, 300], "max_depth": [2, 3, 4], "learning_rate": [0.01, 0.05, 0.1]},
    ),
}

best_estimators = {}
for name, (estimator, grid) in param_grids.items():
    search = GridSearchCV(estimator, grid, cv=cv, scoring="neg_mean_absolute_error", n_jobs=-1)
    search.fit(X_train, y_train)
    best_estimators[name] = search.best_estimator_
    print(f"{name:>17}: best MAE {-search.best_score_:.3f}  params {search.best_params_}")


## Step 9 — Bias/variance diagnostics

**Learning curve** for the tuned RandomForest: training vs CV score as
training-set size grows. A large, non-closing gap between the two lines
means high variance (overfitting, would benefit from more data or more
regularization); both curves converging to a low score means high bias
(underfitting, the model is too constrained).

**Regularization sweep** for Ridge: training vs CV score across `alpha`.
Low alpha (little regularization) overfits -- high train score, lower CV
score. High alpha underfits both. The gap between the two curves *is* the
bias-variance tradeoff, directly visualized.

In [ ]:
train_sizes, train_scores, val_scores = learning_curve(
    best_estimators["RandomForest"],
    X_train, y_train,
    cv=cv,
    scoring="neg_mean_absolute_error",
    train_sizes=np.linspace(0.2, 1.0, 8),
    random_state=RANDOM_STATE,
)

fig, ax = plt.subplots(figsize=(7, 5))
ax.plot(train_sizes, -train_scores.mean(axis=1), "o-", label="Training MAE")
ax.plot(train_sizes, -val_scores.mean(axis=1), "o-", label="CV MAE")
ax.fill_between(train_sizes, -train_scores.mean(axis=1) - train_scores.std(axis=1),
                 -train_scores.mean(axis=1) + train_scores.std(axis=1), alpha=0.15)
ax.fill_between(train_sizes, -val_scores.mean(axis=1) - val_scores.std(axis=1),
                 -val_scores.mean(axis=1) + val_scores.std(axis=1), alpha=0.15)
ax.set_xlabel("Training examples")
ax.set_ylabel("MAE")
ax.set_title("Learning curve -- tuned RandomForest")
ax.legend()
plt.show()


In [ ]:
alphas = np.logspace(-3, 3, 13)
train_scores, val_scores = validation_curve(
    Ridge(random_state=RANDOM_STATE), X_train, y_train,
    param_name="alpha", param_range=alphas,
    cv=cv, scoring="neg_mean_absolute_error",
)

fig, ax = plt.subplots(figsize=(7, 5))
ax.plot(alphas, -train_scores.mean(axis=1), "o-", label="Training MAE")
ax.plot(alphas, -val_scores.mean(axis=1), "o-", label="CV MAE")
ax.set_xscale("log")
ax.set_xlabel("alpha (regularization strength)")
ax.set_ylabel("MAE")
ax.set_title("Regularization sweep -- Ridge")
ax.legend()
plt.show()


## Step 10 — Final held-out evaluation

Pick the model with the best CV MAE from Step 8, refit it on the full
training set, and score it once against `X_test`/`y_test` -- the split
from Step 5 that nothing above has touched. This is the honest,
un-tuned-on number.

In [ ]:
cv_scores = {name: -cross_val_score(est, X_train, y_train, cv=cv, scoring="neg_mean_absolute_error").mean()
             for name, est in best_estimators.items()}
best_name = min(cv_scores, key=cv_scores.get)
best_model = best_estimators[best_name]

print(f"Best model: {best_name} (CV MAE {cv_scores[best_name]:.3f})")

best_model.fit(X_train, y_train)
y_pred = best_model.predict(X_test)

print(f"Held-out MAE:  {mean_absolute_error(y_test, y_pred):.3f}")
print(f"Held-out RMSE: {mean_squared_error(y_test, y_pred) ** 0.5:.3f}")
print(f"Held-out R^2:  {r2_score(y_test, y_pred):.3f}")

fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(y_test, y_pred, alpha=0.6)
ax.plot([1, 5], [1, 5], "k--", linewidth=1)
ax.set_xlabel("Actual rating")
ax.set_ylabel("Predicted rating")
ax.set_title(f"Held-out predictions -- {best_name}")
plt.show()


## Step 11 — Permutation importance

Which features the best model actually relies on, measured on the
held-out set: shuffle one feature column at a time and see how much MAE
gets worse. Unlike a Ridge coefficient or a tree's built-in
`feature_importances_`, this works the same way regardless of model type
and reflects real predictive contribution rather than raw coefficient
magnitude.

In [ ]:
perm = permutation_importance(
    best_model, X_test, y_test,
    scoring="neg_mean_absolute_error",
    n_repeats=30, random_state=RANDOM_STATE,
)

importance_df = pd.DataFrame({
    "feature": FEATURE_COLS,
    "importance": perm.importances_mean,
    "std": perm.importances_std,
}).sort_values("importance", ascending=True)

fig, ax = plt.subplots(figsize=(8, 6))
ax.barh(importance_df["feature"], importance_df["importance"], xerr=importance_df["std"])
ax.set_xlabel("Increase in MAE when shuffled")
ax.set_title(f"Permutation importance -- {best_name}")
plt.tight_layout()
plt.show()


## Step 12 — Nested cross-validation

The negative held-out R^2 in Step 10 isn't necessarily a bad model -- it's
one 46-row draw, and picking "best model" by comparing CV scores across
three tuned candidates adds a little optimism on top of that. **Nested
CV** fixes both: an outer 5-fold loop holds out a fold as test data, an
inner 5-fold loop tunes hyperparameters using only the remaining data (so
tuning never sees the fold it's about to be scored on), and this repeats
until every row has been in the outer test fold exactly once. The result
is an MAE averaged over 5 independent test folds instead of trusting a
single split -- the honest way to compare model families here, not a
replacement for Step 8's tuning (that best-hyperparameter search still
feeds the final model you'd actually ship).

In [ ]:
outer_cv = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
inner_cv = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

nested_mae = {}
for name, (estimator, grid) in param_grids.items():
    search = GridSearchCV(estimator, grid, cv=inner_cv, scoring="neg_mean_absolute_error", n_jobs=-1)
    scores = -cross_val_score(search, X, y, cv=outer_cv, scoring="neg_mean_absolute_error", n_jobs=-1)
    nested_mae[name] = scores
    print(f"{name:>17}: nested MAE {scores.mean():.3f} +/- {scores.std():.3f}")

dummy_nested = -cross_val_score(DummyRegressor(strategy="mean"), X, y, cv=outer_cv, scoring="neg_mean_absolute_error")
print(f"{'Dummy (mean)':>17}: nested MAE {dummy_nested.mean():.3f} +/- {dummy_nested.std():.3f}")
